# OAT preliminary — location analysis (prompt 28 T1.5)

> **PRELIMINARY (OAT-based; single-tier evidence, blind to interactions; pair grids in flight).**
> OAT sweeps cannot see interactions — the wind pair's known sub-additivity is exactly the class
> of effect this analysis is structurally blind to. Verdicts here are v0 and are superseded by
> `LOCATION.md` v1 once the priority pair grids land (T2).

Data: `waves/sweep_B` + `waves/sweep_C` (54 rows each, complete). `stage2_backfill_C` is folded
in **iff** its `objectives.csv` exists at execution time; otherwise nuclear/pv curves are flagged
truncated ("old level set"). Estimators pinned by prompt 28 (v3, post-3-expert-review).

In [1]:
import json, os, sys, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

CAMPAIGN = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, CAMPAIGN)
from tiers import TIERS, STAGE2_LATTICE, load_gen_pmax

OBJ_COLS = ["load_shed_mwh", "true_curtailment_mwh", "total_cost_raw_usd",
            "total_cost_less_synthetic_usd", "reserve_shortfall_mwh", "thermal_starts"]
OBJ_LABEL = {"load_shed_mwh": "load shed [MWh]", "true_curtailment_mwh": "curtailment (g2) [MWh]",
             "total_cost_raw_usd": "total cost, raw [$]",
             "total_cost_less_synthetic_usd": "total cost less synthetic [$]",
             "reserve_shortfall_mwh": "reserve shortfall [MWh]", "thermal_starts": "thermal starts [-]"}
# Noise floors: pi_0911 §3.5 (the CANONICAL record). Reserve shortfall and
# thermal starts have NO established floor (replicate study D3 open) — they
# use the range-and-replication criterion, never a borrowed floor.
FLOORS = {"load_shed_mwh": 3000.0, "true_curtailment_mwh": 5000.0,
          "total_cost_raw_usd": 0.5e6, "total_cost_less_synthetic_usd": 0.5e6,
          "reserve_shortfall_mwh": None, "thermal_starts": None}
FLOORED = [o for o in OBJ_COLS if FLOORS[o] is not None]

MC_SEED, MC_N = 20260920, 10_000   # pinned: floor ≡ 1σ Gaussian, N=10,000, fixed seed

# Total-PEM-MW conversion, pinned — implemented ONCE, used everywhere.
# Nameplates from load_gen_pmax() (NOT tier['gen_pmax']: build_tiers omits it
# for nuclear — KeyError trap).
_PMAX = load_gen_pmax()
NAMEPLATE = {t: sum(_PMAX[m] for m in TIERS[t]["members"]) for t in TIERS}
def mw_of(tier, omega):
    return np.asarray(omega, dtype=float) * NAMEPLATE[tier]

SORTED_TIERS = sorted(TIERS)
print({t: NAMEPLATE[t] for t in SORTED_TIERS})

def boldify(ax):
    for s in ax.spines.values():
        s.set_linewidth(1.5)
    ax.xaxis.label.set_fontweight("bold"); ax.yaxis.label.set_fontweight("bold")
    ax.title.set_fontweight("bold")
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")
os.makedirs("figs", exist_ok=True)

{'nuclear': 400.0, 'pv': 340.5, 'tail': 251.5, 'wind_122': 713.5, 'wind_303': 847.0, 'wind_317': 799.1}


## Provenance

Wind-pair equivalence (`contour_303x317_C` axes vs `STAGE2_LATTICE`) and back-fill availability,
recorded here per the prompt (old CSV stores raw-linspace doubles; exact `==` fails at one level).

In [2]:
# --- contour_303x317_C ≡ STAGE2_LATTICE equivalence (allclose, atol=1e-9) ----
cdm = pd.read_csv(os.path.join(CAMPAIGN, "waves", "contour_303x317_C", "design_matrix.csv"))
lattice = np.asarray(STAGE2_LATTICE, dtype=float)
equiv = {}
for col in ("wind_303_omega", "wind_317_omega"):
    levels = np.sort(cdm[col].dropna().unique())
    assert len(levels) == 9, (col, len(levels))
    equiv[col] = {"allclose_1e-9": bool(np.allclose(levels, lattice, atol=1e-9)),
                  "exact_equal": bool(np.array_equal(levels, lattice)),
                  "max_abs_dev": float(np.max(np.abs(levels - lattice)))}
    assert equiv[col]["allclose_1e-9"], f"{col}: contour axis is NOT the stage-2 lattice"
print("contour_303x317_C ≡ STAGE2_LATTICE:", json.dumps(equiv, indent=2))
print("=> any old↔new join on omega must round to 10 decimals first (exact == fails).")

# --- back-fill availability ---------------------------------------------------
BACKFILL_OBJ = os.path.join(CAMPAIGN, "waves", "stage2_backfill_C", "objectives.csv")
BACKFILL_LANDED = os.path.isfile(BACKFILL_OBJ)
print("stage2_backfill_C objectives.csv landed:", BACKFILL_LANDED)
if not BACKFILL_LANDED:
    print("=> scenario-C nuclear/pv curves are TRUNCATED to the old level set "
          "(nuclear ≤ 0.5, pv ≤ 0.8); window widens to ~[42.4, 251.5] MW once it lands.")

contour_303x317_C ≡ STAGE2_LATTICE: {
  "wind_303_omega": {
    "allclose_1e-9": true,
    "exact_equal": false,
    "max_abs_dev": 1.1102230246251565e-16
  },
  "wind_317_omega": {
    "allclose_1e-9": true,
    "exact_equal": false,
    "max_abs_dev": 1.1102230246251565e-16
  }
}
=> any old↔new join on omega must round to 10 decimals first (exact == fails).
stage2_backfill_C objectives.csv landed: False
=> scenario-C nuclear/pv curves are TRUNCATED to the old level set (nuclear ≤ 0.5, pv ≤ 0.8); window widens to ~[42.4, 251.5] MW once it lands.


In [3]:
# --- load sweeps into long form ----------------------------------------------
def load_sweep(scenario):
    wdir = os.path.join(CAMPAIGN, "waves", f"sweep_{scenario}")
    dm = pd.read_csv(os.path.join(wdir, "design_matrix.csv"))
    ob = pd.read_csv(os.path.join(wdir, "objectives.csv"))
    df = dm.merge(ob[["index"] + OBJ_COLS], on="index", validate="1:1")
    recs = []
    for _, r in df.iterrows():
        active = [t for t in TIERS if pd.notna(r[f"{t}_omega"])]
        assert len(active) == 1, (scenario, r["index"], active)
        t = active[0]
        recs.append({"tier": t, "omega": float(r[f"{t}_omega"]),
                     "T_mw": float(mw_of(t, r[f"{t}_omega"])),
                     **{o: float(r[o]) for o in OBJ_COLS}})
    out = pd.DataFrame(recs).sort_values(["tier", "T_mw"]).reset_index(drop=True)
    assert len(out) == 54 and out.groupby("tier").size().eq(9).all()
    return out

DATA = {"B": load_sweep("B"), "C": load_sweep("C")}
TRUNCATED = {"nuclear": "old cap ω ≤ 0.5 (≤ 200 MW)", "pv": "old top ω ≤ 0.8 (≤ 272 MW)"}

if BACKFILL_LANDED:
    bdm = pd.read_csv(os.path.join(CAMPAIGN, "waves", "stage2_backfill_C", "design_matrix.csv"))
    bob = pd.read_csv(BACKFILL_OBJ)
    bdf = bdm.merge(bob[["index"] + OBJ_COLS], on="index", validate="1:1")
    recs = []
    for _, r in bdf.iterrows():
        t = [t for t in TIERS if pd.notna(r[f"{t}_omega"])][0]
        recs.append({"tier": t, "omega": float(r[f"{t}_omega"]),
                     "T_mw": float(mw_of(t, r[f"{t}_omega"])),
                     **{o: float(r[o]) for o in OBJ_COLS}})
    DATA["C"] = (pd.concat([DATA["C"], pd.DataFrame(recs)])
                 .sort_values(["tier", "T_mw"]).reset_index(drop=True))
    TRUNCATED = {}
    print(f"folded {len(recs)} backfill rows into scenario C")
for s, df in DATA.items():
    print(s, df.groupby("tier").size().to_dict())

B {'nuclear': 9, 'pv': 9, 'tail': 9, 'wind_122': 9, 'wind_303': 9, 'wind_317': 9}
C {'nuclear': 9, 'pv': 9, 'tail': 9, 'wind_122': 9, 'wind_303': 9, 'wind_317': 9}


## Common window & per-tier interpolants

Piecewise-linear `g_i(T)` per tier over total PEM MW (pinned conversion). The common window is
the intersection of the per-tier T ranges; the prompt's audited value is ≈ [42.4, 200] MW,
set by wind_303's ω-floor and nuclear's old cap.

In [4]:
def tier_range(df, t):
    sub = df[df.tier == t]
    return float(sub.T_mw.min()), float(sub.T_mw.max())

WINDOW = {}
for s, df in DATA.items():
    lo = max(tier_range(df, t)[0] for t in SORTED_TIERS)
    hi = min(tier_range(df, t)[1] for t in SORTED_TIERS)
    WINDOW[s] = (lo, hi)
    print(f"scenario {s}: common window = [{lo:.2f}, {hi:.2f}] MW")
    for t in SORTED_TIERS:
        sub = df[df.tier == t]
        n_in = int(((sub.T_mw >= lo) & (sub.T_mw <= hi)).sum())
        note = f"  << TRUNCATED: {TRUNCATED[t]}" if t in TRUNCATED else ""
        flag = "  << <4 in-window points — pairwise/slope supplements apply" if n_in < 4 else ""
        print(f"  {t:9s} own range [{sub.T_mw.min():7.2f}, {sub.T_mw.max():7.2f}] MW, "
              f"{n_in}/9 in-window{flag}{note}")
print("\nRegions the 6-way comparison CANNOT see: T < ~42.4 MW and T > 200 MW —")
print("the deep-retrofit wind region (up to 847 MW at ω=1 for wind_303) is outside it.")

def g_of(df, t, obj):
    sub = df[df.tier == t]
    x, y = sub.T_mw.to_numpy(), sub[obj].to_numpy()
    return lambda T, x=x, y=y: np.interp(T, x, y)

scenario B: common window = [42.35, 200.00] MW
  nuclear   own range [  20.00,  200.00] MW, 8/9 in-window  << TRUNCATED: old cap ω ≤ 0.5 (≤ 200 MW)
  pv        own range [   6.81,  272.40] MW, 4/9 in-window  << TRUNCATED: old top ω ≤ 0.8 (≤ 272 MW)
  tail      own range [   5.03,  251.50] MW, 5/9 in-window
  wind_122  own range [  35.68,  713.50] MW, 1/9 in-window  << <4 in-window points — pairwise/slope supplements apply
  wind_303  own range [  42.35,  847.00] MW, 2/9 in-window  << <4 in-window points — pairwise/slope supplements apply
  wind_317  own range [  39.96,  799.10] MW, 1/9 in-window  << <4 in-window points — pairwise/slope supplements apply
scenario C: common window = [42.35, 200.00] MW
  nuclear   own range [  20.00,  200.00] MW, 8/9 in-window  << TRUNCATED: old cap ω ≤ 0.5 (≤ 200 MW)
  pv        own range [   6.81,  272.40] MW, 4/9 in-window  << TRUNCATED: old top ω ≤ 0.8 (≤ 272 MW)
  tail      own range [   5.03,  251.50] MW, 5/9 in-window
  wind_122  own range [  35.68

## Location spread `L(T)` with the correct range floor

`L(T) = max_i g_i(T) − min_i g_i(T)` on a 1 MW grid within the common window. Its floor is the
95th percentile of the **range of 6 draws** from the per-value noise model (floor ≡ 1σ Gaussian,
MC, N=10,000, seed 20260920) — ratioing L to the *single-value* floor would manufacture ~2.5×
fake signal. Primary summary = median-over-T of `L/floor_L`; max-over-T carries a multiplicity
caveat (it is the maximum of many correlated reads).

In [5]:
rng = np.random.default_rng(MC_SEED)
_draws6 = rng.normal(size=(MC_N, 6)); Q6 = float(np.percentile(np.ptp(_draws6, axis=1), 95))
_draws2 = rng.normal(size=(MC_N, 2)); Q2 = float(np.percentile(np.ptp(_draws2, axis=1), 95))
print(f"range-floor factors (95th pct of range of N iid N(0,1)): n=6: {Q6:.3f}σ, n=2: {Q2:.3f}σ")
print(f"vs single-value floor: the n=6 range floor is {Q6:.2f}× — ~4× as the prompt states")

spread_rows = []
SPREAD_CURVES = {}
for s, df in DATA.items():
    lo, hi = WINDOW[s]
    Tgrid = np.arange(np.ceil(lo), np.floor(hi) + 1.0)   # 1 MW T-grid (pinned)
    for obj in OBJ_COLS:
        G = np.vstack([g_of(df, t, obj)(Tgrid) for t in SORTED_TIERS])
        L = G.max(axis=0) - G.min(axis=0)
        SPREAD_CURVES[(s, obj)] = (Tgrid, L)
        row = {"scenario": s, "objective": obj, "L_median": float(np.median(L)),
               "L_max": float(L.max()), "T_at_max": float(Tgrid[np.argmax(L)])}
        if FLOORS[obj] is not None:
            floor_L = FLOORS[obj] * Q6
            row.update({"floor_L": floor_L,
                        "ratio_median": float(np.median(L / floor_L)),
                        "ratio_max": float(L.max() / floor_L)})
        spread_rows.append(row)
spread = pd.DataFrame(spread_rows)
spread.to_csv("oat_spread_summary.csv", index=False)
with pd.option_context("display.width", 160):
    print(spread.round(3).to_string(index=False))

range-floor factors (95th pct of range of N iid N(0,1)): n=6: 4.040σ, n=2: 2.792σ
vs single-value floor: the n=6 range floor is 4.04× — ~4× as the prompt states
scenario                     objective     L_median        L_max  T_at_max     floor_L  ratio_median  ratio_max
       B                 load_shed_mwh     6381.815    10487.019     173.0   12119.260         0.527      0.865
       B          true_curtailment_mwh   149996.603   234907.765     200.0   20198.767         7.426     11.630
       B            total_cost_raw_usd 17856230.210 25828223.299      43.0 2019876.746         8.840     12.787
       B total_cost_less_synthetic_usd 17629591.016 25754274.960      43.0 2019876.746         8.728     12.750
       B         reserve_shortfall_mwh    26927.754    40266.354     200.0         NaN           NaN        NaN
       B                thermal_starts      291.983      352.495     200.0         NaN           NaN        NaN
       C                 load_shed_mwh     7354.672    

## Supplements for thin in-window tiers

Two supplements, since wind_122/wind_317 (and wind_303) have < 4 in-window points:
(a) the full **pairwise-window matrix** — each pair compared on its *own* overlap window with the
n=2 range floor; (b) **matched-T marginal slopes** `dg_i/dT` (OLS per tier; in-window points when
≥ 2, else the tier's full own range, flagged `full-range`).

In [6]:
pair_rows = []
for s, df in DATA.items():
    for obj in FLOORED:
        floor2 = FLOORS[obj] * Q2
        for a, b in itertools.combinations(SORTED_TIERS, 2):
            lo = max(tier_range(df, a)[0], tier_range(df, b)[0])
            hi = min(tier_range(df, a)[1], tier_range(df, b)[1])
            Tg = np.arange(np.ceil(lo), np.floor(hi) + 1.0)
            D = g_of(df, a, obj)(Tg) - g_of(df, b, obj)(Tg)
            pair_rows.append({"scenario": s, "objective": obj, "pair": f"{a}|{b}",
                              "window_lo": lo, "window_hi": hi,
                              "median_absdiff_over_floor2": float(np.median(np.abs(D)) / floor2),
                              "max_absdiff_over_floor2": float(np.max(np.abs(D)) / floor2),
                              "median_signed_diff": float(np.median(D))})
pairwise = pd.DataFrame(pair_rows)
pairwise.to_csv("oat_pairwise_matrix.csv", index=False)
for s in DATA:
    piv = pairwise[pairwise.scenario == s].pivot(index="pair", columns="objective",
                                                 values="median_absdiff_over_floor2")
    print(f"\nscenario {s} — median |g_a − g_b| / floor_L2 on each pair's own window:")
    print(piv.round(2).to_string())

slope_rows = []
for s, df in DATA.items():
    lo, hi = WINDOW[s]
    for obj in OBJ_COLS:
        for t in SORTED_TIERS:
            sub = df[df.tier == t]
            inw = sub[(sub.T_mw >= lo) & (sub.T_mw <= hi)]
            basis, src = (inw, "in-window") if len(inw) >= 2 else (sub, "full-range")
            slope = float(np.polyfit(basis.T_mw, basis[obj], 1)[0])
            slope_rows.append({"scenario": s, "objective": obj, "tier": t,
                               "dg_dT": slope, "basis": src, "n_pts": len(basis)})
slopes = pd.DataFrame(slope_rows)
slopes.to_csv("oat_marginal_slopes.csv", index=False)
for s in DATA:
    piv = slopes[slopes.scenario == s].pivot(index="tier", columns="objective", values="dg_dT")
    print(f"\nscenario {s} — marginal slopes dg/dT (per MW of PEM):")
    print(piv.round(3).to_string())


scenario B — median |g_a − g_b| / floor_L2 on each pair's own window:
objective          load_shed_mwh  total_cost_less_synthetic_usd  total_cost_raw_usd  true_curtailment_mwh
pair                                                                                                     
nuclear|pv                  0.22                          13.50               12.79                  5.37
nuclear|tail                0.47                          11.37               11.44                  6.34
nuclear|wind_122            0.75                          12.82               13.05                  4.36
nuclear|wind_303            0.67                          12.35               12.21                  3.99
nuclear|wind_317            0.71                          12.53               12.69                  2.60
pv|tail                     0.23                           2.41                1.53                  0.87
pv|wind_122                 0.83                           0.25                1.

## Variance decomposition — matched-basis nested ANCOVA

The naive version is degenerate (per-tier 9-point piecewise fits are saturated, R² ≡ 1). Pinned
basis: **model 0** = natural cubic spline in T, **4 df**, knots at the pooled-T quartiles within
the common window (boundary knots at the window edges; natural ⇒ linear beyond them);
**model 1** = + tier intercepts (pure separation); **model 2** = + tier×T (shape). Fit on the
**full pooled 54 points** per scenario — restricting to in-window points leaves wind_122/wind_317
with a single point each, making the tier×T terms unidentifiable (analysis decision, stated).
F uses the §3.5 floor as σ (floor ≡ 1σ Gaussian); floorless objectives report increments only.

In [7]:
def ns_basis(x, boundary, interior):
    # natural cubic regression spline (ESL §5.2.1): columns [x, N3..N_{K}] with
    # K = 2 boundary + len(interior) knots -> len(interior)+1 columns + x = 4 df here
    knots = np.sort(np.r_[boundary[0], interior, boundary[1]])
    K = len(knots)
    def d(k, xx):
        num = (np.maximum(xx - knots[k], 0.0) ** 3
               - np.maximum(xx - knots[-1], 0.0) ** 3)
        return num / (knots[-1] - knots[k])
    cols = [x] + [d(k, x) - d(K - 2, x) for k in range(K - 2)]
    return np.column_stack(cols)

def fit_sse(X, y):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    r = y - X @ beta
    return float(r @ r), X.shape[1]

anc_rows = []
for s, df in DATA.items():
    lo, hi = WINDOW[s]
    pooled_inwin = df[(df.T_mw >= lo) & (df.T_mw <= hi)]
    interior = np.percentile(pooled_inwin.T_mw, [25, 50, 75])   # pinned knots
    T, n = df.T_mw.to_numpy(), len(df)
    NS = ns_basis(T, (lo, hi), interior)
    dummies = pd.get_dummies(df.tier, drop_first=True).to_numpy(dtype=float)
    X0 = np.column_stack([np.ones(n), NS])
    X1 = np.column_stack([X0, dummies])
    X2 = np.column_stack([X1, dummies * T[:, None]])
    for obj in OBJ_COLS:
        y = df[obj].to_numpy()
        sst = float(((y - y.mean()) ** 2).sum())
        out = {"scenario": s, "objective": obj}
        sses, ps = [], []
        for tag, X in (("m0_total", X0), ("m1_+intercepts", X1), ("m2_+tierxT", X2)):
            sse, p = fit_sse(X, y)
            r2adj = 1.0 - (sse / (n - p)) / (sst / (n - 1))
            out[f"{tag}_R2adj"] = round(r2adj, 4)
            sses.append(sse); ps.append(p)
        for name, (s0, s1), (p0, p1) in (("intercepts", sses[:2], ps[:2]),
                                          ("tierxT", sses[1:], ps[1:])):
            ddf = p1 - p0
            out[f"d_{name}_dR2adj"] = round(out[f"m{'1_+intercepts' if name=='intercepts' else '2_+tierxT'}_R2adj"]
                                            - out[f"m{'0_total' if name=='intercepts' else '1_+intercepts'}_R2adj"], 4)
            if FLOORS[obj] is not None:
                F = ((s0 - s1) / ddf) / FLOORS[obj] ** 2
                out[f"F_{name}"] = round(F, 2)
                out[f"p_{name}"] = float(stats.chi2.sf((s0 - s1) / FLOORS[obj] ** 2, ddf))
        anc_rows.append(out)
ancova = pd.DataFrame(anc_rows)
ancova.to_csv("oat_ancova.csv", index=False)
with pd.option_context("display.width", 220, "display.max_columns", 40):
    print(ancova.to_string(index=False))

scenario                     objective  m0_total_R2adj  m1_+intercepts_R2adj  m2_+tierxT_R2adj  d_intercepts_dR2adj  F_intercepts  p_intercepts  d_tierxT_dR2adj  F_tierxT  p_tierxT
       B                 load_shed_mwh          0.7722                0.9135            0.9678               0.1413         13.66  2.340324e-13           0.0543      4.73  0.000253
       B          true_curtailment_mwh          0.9033                0.9735            0.9873               0.0702       1665.45  0.000000e+00           0.0138    313.69  0.000000
       B            total_cost_raw_usd          0.6090                0.9281            0.9975               0.3191       1662.37  0.000000e+00           0.0694    318.67  0.000000
       B total_cost_less_synthetic_usd          0.5238                0.9202            0.9982               0.3964       1685.77  0.000000e+00           0.0780    292.48  0.000000
       B         reserve_shortfall_mwh          0.8031                0.9456            0.9838 

## Site-ranking stability (across the window; B vs C)

Tier ranks of `g_i(T)` at matched T. The nuclear B/C comparison is valid **only ≤ 200 MW** —
B-scenario back-fills above the old cap are deferred by decision.

In [8]:
RANK_TS = [50.0, 100.0, 150.0, 200.0]
rank_tabs = {}
for s, df in DATA.items():
    lo, hi = WINDOW[s]
    ts = [t for t in RANK_TS if lo <= t <= hi]
    tab = {}
    for obj in OBJ_COLS:
        vals = {t: [float(g_of(df, t, obj)(T)) for T in ts] for t in SORTED_TIERS}
        tab[obj] = pd.DataFrame(vals, index=[f"T={T:.0f}" for T in ts]).rank(axis=1).astype(int)
    rank_tabs[s] = tab
for obj in OBJ_COLS:
    print(f"\n--- {obj}: rank of g_i(T), 1 = lowest (B | C) ---")
    print(pd.concat({"B": rank_tabs["B"][obj], "C": rank_tabs["C"][obj]}, axis=1).to_string())

stab_rows = []
for s in DATA:
    for obj in OBJ_COLS:
        r = rank_tabs[s][obj]
        tau_w = stats.kendalltau(r.iloc[0], r.iloc[-1]).statistic  # window ends
        tau_bc = stats.kendalltau(rank_tabs["B"][obj].iloc[-1],
                                  rank_tabs["C"][obj].iloc[-1]).statistic
        stab_rows.append({"scenario": s, "objective": obj,
                          "tau_T50_vs_T200": round(float(tau_w), 3),
                          "tau_BvsC_at_T200": round(float(tau_bc), 3)})
stability = pd.DataFrame(stab_rows)
stability.to_csv("oat_rank_stability.csv", index=False)
print("\n", stability.to_string(index=False))


--- load_shed_mwh: rank of g_i(T), 1 = lowest (B | C) ---
            B                                          C                                   
      nuclear pv tail wind_122 wind_303 wind_317 nuclear pv tail wind_122 wind_303 wind_317
T=50        6  3    1        2        5        4       6  5    4        3        1        2
T=100       6  5    4        1        3        2       6  5    4        2        1        3
T=150       6  5    4        1        3        2       6  5    4        1        2        3
T=200       5  6    4        2        3        1       6  5    4        1        3        2

--- true_curtailment_mwh: rank of g_i(T), 1 = lowest (B | C) ---
            B                                          C                                   
      nuclear pv tail wind_122 wind_303 wind_317 nuclear pv tail wind_122 wind_303 wind_317
T=50        2  6    5        4        1        3       2  5    6        4        1        3
T=100       2  5    6        4        1        


 scenario                     objective  tau_T50_vs_T200  tau_BvsC_at_T200
       B                 load_shed_mwh            0.067             0.733
       B          true_curtailment_mwh            0.867             1.000
       B            total_cost_raw_usd            0.600             1.000
       B total_cost_less_synthetic_usd            0.200             0.867
       B         reserve_shortfall_mwh            0.733             0.867
       B                thermal_starts            0.333             0.867
       C                 load_shed_mwh            0.600             0.733
       C          true_curtailment_mwh            1.000             1.000
       C            total_cost_raw_usd            0.600             1.000
       C total_cost_less_synthetic_usd            0.333             0.867
       C         reserve_shortfall_mwh            0.467             0.867
       C                thermal_starts            0.600             0.867


## Floorless objectives (reserve shortfall, thermal starts)

No established floor (replicate study D3 open) — never borrow shed's. Verdict only on effects
≥ 10% of the objective's own observed range AND consistent sign across ≥ 2 independent
comparisons (here: the B and C sweeps); otherwise **not resolvable (no floor; pending D3)**.

In [9]:
floorless_rows = []
for obj in [o for o in OBJ_COLS if FLOORS[o] is None]:
    own_range = float(pd.concat([DATA["B"][obj], DATA["C"][obj]]).pipe(lambda x: x.max() - x.min()))
    for a, b in itertools.combinations(SORTED_TIERS, 2):
        med = {}
        for s, df in DATA.items():
            lo = max(tier_range(df, a)[0], tier_range(df, b)[0])
            hi = min(tier_range(df, a)[1], tier_range(df, b)[1])
            Tg = np.arange(np.ceil(lo), np.floor(hi) + 1.0)
            med[s] = float(np.median(g_of(df, a, obj)(Tg) - g_of(df, b, obj)(Tg)))
        big = all(abs(m) >= 0.10 * own_range for m in med.values())
        same_sign = np.sign(med["B"]) == np.sign(med["C"]) != 0
        floorless_rows.append({"objective": obj, "pair": f"{a}|{b}",
                               "own_range": own_range, "med_diff_B": med["B"],
                               "med_diff_C": med["C"],
                               "resolvable": bool(big and same_sign)})
floorless = pd.DataFrame(floorless_rows)
floorless.to_csv("oat_floorless.csv", index=False)
print(floorless.round(1).to_string(index=False))
for obj in floorless.objective.unique():
    n = int(floorless[floorless.objective == obj].resolvable.sum())
    print(f"{obj}: {n}/15 pairs meet the >=10%-of-range + consistent-sign criterion")

            objective              pair  own_range  med_diff_B  med_diff_C  resolvable
reserve_shortfall_mwh        nuclear|pv   119639.7     -6432.4     -4455.4       False
reserve_shortfall_mwh      nuclear|tail   119639.7     -2278.0       656.4       False
reserve_shortfall_mwh  nuclear|wind_122   119639.7     15399.7     21572.7        True
reserve_shortfall_mwh  nuclear|wind_303   119639.7      5744.1      9132.4       False
reserve_shortfall_mwh  nuclear|wind_317   119639.7     19684.0     18375.7        True
reserve_shortfall_mwh           pv|tail   119639.7      6599.2      6782.0       False
reserve_shortfall_mwh       pv|wind_122   119639.7     26554.5     30921.4        True
reserve_shortfall_mwh       pv|wind_303   119639.7     13965.0     15832.4        True
reserve_shortfall_mwh       pv|wind_317   119639.7     30680.0     30222.5        True
reserve_shortfall_mwh     tail|wind_122   119639.7     22366.6     27605.3        True
reserve_shortfall_mwh     tail|wind_303   1

## Figures (house style: bold axes)

In [10]:
colors = dict(zip(SORTED_TIERS, plt.cm.tab10.colors))
for s, df in DATA.items():
    lo, hi = WINDOW[s]
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, obj in zip(axes.flat, OBJ_COLS):
        for t in SORTED_TIERS:
            sub = df[df.tier == t]
            ls = "--" if t in TRUNCATED else "-"
            ax.plot(sub.T_mw, sub[obj], marker="o", ms=4, ls=ls, color=colors[t],
                    label=t + (" (trunc.)" if t in TRUNCATED else ""))
        ax.axvspan(lo, hi, color="0.85", alpha=0.5, zorder=0)
        ax.set_xlabel("total PEM MW"); ax.set_ylabel(OBJ_LABEL[obj])
        ax.set_title(f"{obj} — scenario {s}", fontsize=10)
        boldify(ax)
    axes.flat[0].legend(fontsize=8, ncol=2)
    fig.suptitle(f"OAT g_i(T) per tier — scenario {s} (grey band = common window)",
                 fontweight="bold")
    fig.tight_layout()
    fig.savefig(f"figs/oat_curves_{s}.png", dpi=150); plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, obj in zip(axes.flat, FLOORED):
    for s, c in (("B", "tab:blue"), ("C", "tab:red")):
        Tg, L = SPREAD_CURVES[(s, obj)]
        ax.plot(Tg, L / (FLOORS[obj] * Q6), color=c, label=f"scenario {s}")
    ax.axhline(1.0, color="k", lw=1, ls=":"); ax.axhline(2.0, color="k", lw=1, ls="--")
    ax.set_xlabel("total PEM MW"); ax.set_ylabel("L(T) / floor_L")
    ax.set_title(obj, fontsize=10); boldify(ax)
axes.flat[0].legend(fontsize=9)
fig.suptitle("Location spread vs its 6-draw range floor (dashes: ratio = 2)", fontweight="bold")
fig.tight_layout(); fig.savefig("figs/oat_spread_ratio.png", dpi=150); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
x = np.arange(len(OBJ_COLS))
for ax, s in zip(axes, DATA):
    sub = ancova[ancova.scenario == s].set_index("objective").loc[OBJ_COLS]
    ax.bar(x - 0.2, sub["d_intercepts_dR2adj"], 0.4, label="+tier intercepts")
    ax.bar(x + 0.2, sub["d_tierxT_dR2adj"], 0.4, label="+tier×T")
    ax.set_xticks(x); ax.set_xticklabels([o.replace("_", "\n") for o in OBJ_COLS], fontsize=7)
    ax.set_ylabel("ΔR²(adj) over matched spline basis"); ax.set_title(f"scenario {s}")
    boldify(ax)
axes[0].legend()
fig.suptitle("Nested ANCOVA increments (model 0 = natural cubic spline in T, 4 df)",
             fontweight="bold")
fig.tight_layout(); fig.savefig("figs/oat_ancova_increments.png", dpi=150); plt.close(fig)
print("figures written:", sorted(os.listdir("figs")))

figures written: ['oat_ancova_increments.png', 'oat_curves_B.png', 'oat_curves_C.png', 'oat_spread_ratio.png']


In [11]:
# machine-readable summary for LOCATION.md v0
summary = {
    "window": {s: list(WINDOW[s]) for s in DATA},
    "backfill_landed": BACKFILL_LANDED,
    "mc": {"seed": MC_SEED, "n": MC_N, "q6": Q6, "q2": Q2},
    "spread": spread.to_dict("records"),
    "ancova": ancova.to_dict("records"),
    "stability": stability.to_dict("records"),
    "floorless_resolvable": {o: int(floorless[floorless.objective == o].resolvable.sum())
                             for o in floorless.objective.unique()},
    "contour_lattice_equiv": equiv,
}
with open("oat_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("oat_summary.json written")

oat_summary.json written
